In [7]:
import re
from pathlib import Path
from typing import Optional

input_file = Path("TEST_COMBINED.txt")
output_file = Path("output_marlin.gcode")

# Change these if needed, e.g. {"A": "E", "B": "Z"}
AXIS_MAP = {
    "A": "A",
    "B": "B",
}

DROP_CODES = {"G94", "G64", "M30"}   # usually omitted for Marlin
REMAP_CODES = {
    "G01": "G1",
    "M00": "M0",
}

def convert_comment_parens_to_semicolon(line: str) -> str:
    def repl(match):
        text = match.group(1).strip()
        return f"; {text}" if text else ""
    return re.sub(r"\((.*?)\)", repl, line)

def remap_axes(line: str) -> str:
    for src, dst in AXIS_MAP.items():
        if src != dst:
            line = re.sub(rf"\b{src}([+-]?\d+(?:\.\d+)?)\b", rf"{dst}\1", line)
    return line

def convert_line(line: str) -> Optional[str]:
    line = line.rstrip()

    if not line:
        return ""

    # Drop % and o-program lines
    if line.strip() == "%":
        return None
    if re.match(r"^\s*o\d+\s*$", line, re.IGNORECASE):
        return None

    # Remove N-line numbers
    line = re.sub(r"^\s*N\d+\s*", "", line, flags=re.IGNORECASE)

    # Convert comments first
    line = convert_comment_parens_to_semicolon(line)

    # Tokenize into code/non-code chunks
    parts = []
    for token in re.split(r"(\s+)", line):
        if not token or token.isspace():
            parts.append(token)
            continue

        upper = token.upper()

        # Exact code remap
        if upper in REMAP_CODES:
            parts.append(REMAP_CODES[upper])
            continue

        # Drop unsupported / unnecessary modal codes
        if upper in DROP_CODES:
            continue

        parts.append(token)

    line = "".join(parts)

    # Axis remap
    line = remap_axes(line)

    # Clean spacing
    line = re.sub(r"\s+", " ", line).strip()
    line = re.sub(r"\bF\s+([+-]?\d+(?:\.\d+)?)", r"F\1", line)

    # Optional: remove redundant "G1" if line is only comments
    if not line:
        return ""

    return line

def convert_text(text: str) -> str:
    out = []

    # --- Startup ---
    out.append("; --- Startup ---")
    out.append("M413 S1")
    out.append("G28 ; Home all axes")
    out.append("")

    for raw in text.splitlines():
        new_line = convert_line(raw)
        if new_line is None:
            continue
        out.append(new_line)

    return "\n".join(out) + "\n"

if __name__ == "__main__":
    input_file = Path("TEST_COMBINED.txt")
    output_file = Path("output_marlin.gcode")

    converted = convert_text(input_file.read_text(encoding="utf-8", errors="ignore"))
    output_file.write_text(converted, encoding="utf-8")